In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

In [2]:
data = pd.read_csv('/Users/ernestofadel/Desktop/DC_local/Data_files/Total_data.csv',
                   header = 0,
                   names = ['Assay_id', 'Description', 'Document_id', 'Document_year', 'Confidence_score', 'Target_id', 'Pref_name', 'Protein_sequence', 'Uniprot_id', 
                          'Variant_id', 'Molecule_id', 'Canonical_smile', 'Standard_type', 'Standard_value', 'Standard_units', 'pchembl_value',
                          'assay_type', 'assay_organism', 'assay_category', 'assay_tax_id', 'assay_strain', 'assay_tissue', 'assay_cell_type',
                          'assay_subcellular_fraction', 'bao_format'],
                   low_memory=False
                   )

data

,Assay_id,Description,Document_id,Document_year,Confidence_score,Target_id,Pref_name,Protein_sequence,Uniprot_id,Variant_id,...,pchembl_value,assay_type,assay_organism,assay_category,assay_tax_id,assay_strain,assay_tissue,assay_cell_type,assay_subcellular_fraction,bao_format
0,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,NaN,...,5.60,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
1,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,NaN,...,5.05,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
2,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,NaN,...,5.40,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
3,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,NaN,...,4.77,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
4,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,NaN,...,6.75,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1414959,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,9.52,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1414960,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,6.13,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1414961,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,7.14,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1414962,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,7.23,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357


In [3]:
#Removing data with confidence score lower than 9

def remove_low_confidence(data):
    
    mask = (data['Confidence_score'] == 9)

    df =  data[mask]

    return df

#Removing data with missing documentation (no document year)

def remove_missing_doc_year(data):

    mask = pd.isna(data['Document_year'])

    df = data[~mask]

    return df

# Remove missing pChembl Value

def remove_missing_pchembl_value(data):

    mask = pd.isna(data['pchembl_value'])

    df = data[~mask]

    return df

# Filter by assay size

def filter_by_size(data, minAssaySize, maxAssaySize):

    #Count how many times each assay appears in filtered_activities
    count = data['Assay_id'].value_counts()

    valid_ids = count[(count >= minAssaySize) & (count <= maxAssaySize)].index

    df = data[data['Assay_id'].isin(valid_ids)]

    return df

#Removing mutated protein

def remove_mutant(data):
    
    mask = pd.isna(data['Variant_id'])

    data = data[mask]

    exclude_keywords = ['mutant', 'mutation', 'variant']

    pattern = '|'.join(exclude_keywords)

    df = data[~data['Description'].str.contains(pattern, case=False, na=False)]

    return df

# Removing experiments with different pChembl value

def filter_pChembl_value(data): #Verify this function (made by Claude)

    pairs = data.merge(
        data,
        on=['Molecule_id', 'Target_id'],    # same molecule and target
        suffixes=('_1', '_2')
    )

    pairs = pairs[pairs['Assay_id_1'] != pairs['Assay_id_2']]

    pairs = pairs[pairs['Assay_id_1'] < pairs['Assay_id_2']]

    pairs['pchembl_diff'] = abs(pairs['pchembl_value_1'] - pairs['pchembl_value_2'])

    pairs_filtered = pairs[~((pairs['pchembl_diff'] == 0) | (pairs['pchembl_diff'] == 3.0))]

    valid_assays_1 = pairs_filtered[['Molecule_id', 'Target_id', 'Assay_id_1']].rename(
        columns={'Assay_id_1': 'Assay_id'}
    )

    valid_assays_2 = pairs_filtered[['Molecule_id', 'Target_id', 'Assay_id_2']].rename(
        columns={'Assay_id_2': 'Assay_id'}
    )

    valid_measurements = pd.concat([valid_assays_1, valid_assays_2]).drop_duplicates()

    keys = ["Molecule_id", "Target_id", "Assay_id"]

    df_clean = data.merge(
        valid_measurements[keys],
        on=keys,
        how="left",
        indicator=True
        )

    df_clean = df_clean[df_clean["_merge"] == "left_only"].drop(columns=["_merge"])

    return df_clean

# Removing assays with different assay_type

def removing_different_assay_type(data):

    inconsistent_ids = data.groupby('Assay_id')['assay_type'].nunique()
    inconsistent_ids = inconsistent_ids[inconsistent_ids > 1].index

    df_clean = data[~data['Assay_id'].isin(inconsistent_ids)]

    return df_clean

# Removing duplicate papers

def removing_duplicate_paper(data):

    inconsistent_ids = data.groupby(['Document_id', 'Target_id'])['Assay_id'].nunique()
    inconsistent_ids = inconsistent_ids[inconsistent_ids > 1].index

    df_clean = data[~data['Assay_id'].isin(inconsistent_ids)]

    return df_clean


In [6]:
# Extracting Landrum dataset from raw data

Data = remove_low_confidence(data)
Data = remove_missing_pchembl_value(Data)
Data = remove_missing_doc_year(Data)
Data = remove_mutant(Data)
Data = filter_pChembl_value(Data)
Landrum = filter_by_size(Data, 20, 100)

Landrum

,Assay_id,Description,Document_id,Document_year,Confidence_score,Target_id,Pref_name,Protein_sequence,Uniprot_id,Variant_id,...,pchembl_value,assay_type,assay_organism,assay_category,assay_tax_id,assay_strain,assay_tissue,assay_cell_type,assay_subcellular_fraction,bao_format
3,CHEMBL641889,Inhibition of Acyl coenzyme A:cholesterol acyl...,CHEMBL1128039,1995.0,9,CHEMBL4464,Sterol O-acyltransferase 1,MSLRNRLSKSGENPEQDEAQKNFMDTYRNGHITMKQLIAKKRLLAA...,Q61263,NaN,...,5.26,B,Mus musculus,NaN,10090.0,NaN,NaN,Macrophage,NaN,BAO_0000219
5,CHEMBL641889,Inhibition of Acyl coenzyme A:cholesterol acyl...,CHEMBL1128039,1995.0,9,CHEMBL4464,Sterol O-acyltransferase 1,MSLRNRLSKSGENPEQDEAQKNFMDTYRNGHITMKQLIAKKRLLAA...,Q61263,NaN,...,6.35,B,Mus musculus,NaN,10090.0,NaN,NaN,Macrophage,NaN,BAO_0000219
26,CHEMBL645934,Inhibitory activity against acyl coenzyme A:ch...,CHEMBL1127563,1994.0,9,CHEMBL285,Sterol O-acyltransferase 1,MVGEETSLRNRLSRSAENPEQDEAQKNLLDTHRNGHITMKQLIAKK...,O70536,NaN,...,7.82,B,Rattus norvegicus,NaN,10116.0,NaN,NaN,NaN,Microsome,BAO_0000251
27,CHEMBL769366,In vivo antiviral activity (IC50) against HIV-...,CHEMBL1129270,1996.0,9,CHEMBL243,Human immunodeficiency virus type 1 protease,PQVTLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKM...,Q72874,NaN,...,7.38,B,Human immunodeficiency virus 1,NaN,11676.0,NaN,NaN,NaN,NaN,BAO_0000218
41,CHEMBL695233,In vitro inhibitory activity against HIV-1 pro...,CHEMBL1127978,1994.0,9,CHEMBL243,Human immunodeficiency virus type 1 protease,PQVTLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKM...,Q72874,NaN,...,8.70,B,Human immunodeficiency virus 1,NaN,11676.0,NaN,NaN,NaN,NaN,BAO_0000357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
949171,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,NaN,...,8.20,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
949172,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,NaN,...,8.35,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
949173,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,NaN,...,8.60,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
949174,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,NaN,...,8.14,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019


In [5]:
# Extracting Omnivore dataset from raw data

Data = remove_missing_pchembl_value(data)
Data = remove_mutant(Data)
Omnivore = filter_pChembl_value(Data)

Omnivore

,Assay_id,Description,Document_id,Document_year,Confidence_score,Target_id,Pref_name,Protein_sequence,Uniprot_id,Variant_id,...,pchembl_value,assay_type,assay_organism,assay_category,assay_tax_id,assay_strain,assay_tissue,assay_cell_type,assay_subcellular_fraction,bao_format
0,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,NaN,...,5.60,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
1,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,NaN,...,5.05,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
2,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,NaN,...,5.40,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
3,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,NaN,...,4.77,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
4,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,NaN,...,6.75,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1344931,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,9.52,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1344932,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,6.13,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1344933,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,7.14,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1344934,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,NaN,...,7.23,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
